In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import *
from fit_pv import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [5]:
folder_blos = '/home/ulyanov/data/solo/phi/2026/blos_/'
folder_vlos = '/home/ulyanov/data/solo/phi/2026/vlos_/'

In [6]:
import fnmatch
from connect import bob

sftp = bob()

top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

folder = '/home/ulyanov/data/solo/phi/2026/'


for directory in dirs:
    if fnmatch.fnmatch(directory, '2026-06*'):
        for file in sorted(sftp.listdir(top_dir + directory))[:1]:
            if fnmatch.fnmatch(file, '*fdt-alam*C_*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                data, header = process(local_file,
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       #cavity_file=cavity_file,
                                       #flatfield_file=flat_file,
                                       #ghost_file=ghost_file,
                                       distortion_file=distortion_file,
                                       #_realign=True,
                                       _find_center=True,
                                       _demodulate=True,
                                       #_correct_fringes=True,
                                       #_correct_crosstalk=True,
                                       _calc_wavelengths=True,
                                       _mask=True,
                                       )

                #data = rebin(data, 4)

                velocity = header['OBS_VR']

                line_params = fit_line(data, header, lam=0.1, niter=10)

                Wmu = np.nanmedian(-line_params[2] / line_params[3])
                shift = np.nanmedian(line_params[0])
                print(Wmu, shift)

                stop

                #stop

                #Blos, Vlos = classical_estimates(data, header, lam=1e-2, niter=10)

                #file_blos = generate_filename(file, prefix='blos', folder=folder_blos)
                #file_vlos = generate_filename(file, prefix='vlos', folder=folder_vlos)

                #clone_fits(local_file, file_blos, Blos, header)
                #clone_fits(local_file, file_vlos, Vlos, header)

solo_L1_phi-fdt-alam_20260601T020009_V202606201333C_0646010501.fits.gz


TypeError: pvfunc() takes 6 positional arguments but 513 were given

In [8]:
plt.figure(figsize=(10,10))
plt.imshow(line_params[0], 'seismic', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(line_params[1], 'inferno', vmin=0.15, vmax=0.2)
plt.tight_layout()

print(np.nanmedian(line_params[1]))

0.16854802776089808


In [10]:
plt.figure(figsize=(10,10))
plt.imshow(-line_params[2] / line_params[3], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(line_params[4], 'inferno', vmin=0., vmax=1)
plt.tight_layout()

In [8]:
q_V = 6173.341 / 299792458

k, b = np.polyfit(shifts, velocities, 1)

print(k * q_V, b * q_V)

fit = k * shifts + b

plt.figure(figsize=(8,8))
plt.plot(shifts, velocities, '.')
plt.plot(shifts, fit, '--', lw=1)
plt.grid(True)
plt.tight_layout()

0.9988109300231882 0.017524087993963097


In [9]:
plt.figure(figsize=(8,8))
plt.plot(velocities, shifts / (6173.341 / 299792458) - velocities, '.')
plt.grid(True)
plt.tight_layout()

In [10]:
plt.figure(figsize=(8,8))
plt.plot(velocities, widths, '.')
plt.grid(True)
plt.tight_layout()

In [39]:
np.polyfit(shifts, velocities, 1)#[0] * 6173.341 / 299792458

array([48316.29841428,   805.64126712])

In [4]:
q_V = 6173.341 / 299792458
q_V * 200

0.00411840981002931